## LABORATORIO 4.1
## ONE_vs_ALL SIN GRAFICOS 


In [145]:
# importando librerias
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from scipy import optimize

In [146]:
# obteniendo los datos

data = pd.read_csv('penguins.csv', delimiter=',')

# cuantificando los datos
data['island'] = data['island'].map({'Torgersen':1 , 'Biscoe':2, 'Dream':3})
data['sex'] = data['sex'].map({'male':1, 'female':2})
data['species'] = data['species'].map({'Adelie':1, 'Gentoo':2, 'Chinstrap':3})

#imprimiendo cuantos valores nulos existen en cada columna
print(data.isnull().sum())

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


In [147]:
# Eliminando los valores nulos 
data = data.dropna()

print(data.isnull().sum())

species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64


In [148]:
# asignando las columnas respectivas a X y Y
x = data[['island','bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g','sex']].values
y = data['species'].values

print(x.shape)
print(y.shape)

(333, 6)
(333,)


### Funcion de Normalizacion

In [149]:
def normalizacion(x):
    x_norm = x.copy()
    miu = np.zeros(x.shape[1])
    sigma = np.zeros(x.shape[1])

    miu = np.mean(x, axis=0)
    sigma = np.std(x, axis=0)
    x_norm = (x - miu) / sigma

    return x_norm, miu, sigma

In [150]:
# normalizando los datos
x_norm, miu , sigma = normalizacion(x)

print("Medias: ",miu)
print("Sigmas: ",sigma)

Medias:  [   2.2282   43.9928   17.1649  200.967  4207.0571    1.4955]
Sigmas:  [  0.6771   5.4605   1.9663  13.9947 804.0059   0.5   ]


In [151]:
print(x_norm)

[[-1.814  -0.896   0.7807 -1.4268 -0.5685 -0.991 ]
 [-1.814  -0.8228  0.1196 -1.0695 -0.5063  1.009 ]
 [-1.814  -0.6763  0.4247 -0.4264 -1.1904  1.009 ]
 ...
 [ 1.1399  1.0269  0.5264 -0.5693 -0.5374 -0.991 ]
 [ 1.1399  1.2466  0.9333  0.6455 -0.1332 -0.991 ]
 [ 1.1399  1.1368  0.7807 -0.212  -0.5374  1.009 ]]


### Función Sigmoide

In [152]:
def sigmoid(z):

    z = np.array(z)
    g = np.array(z.shape)

    

    g = 1/(1 + np.exp(-z))

    return g

### Función de Costo

In [153]:
def costo(theta, x, y, _lambda):
    m = y.size
    # inicializando el gradiente
    grad = np.zeros(theta.shape)

    h = sigmoid(x.dot(theta.T))

    temp = theta.copy()
    temp[0] = 0

    J = (1 / m) * np.sum(-y.dot(np.log(h)) - (1 - y).dot(np.log(1 - h))) + (_lambda / (2 * m)) * np.sum(np.square(temp))

    grad = (1 / m) * (h - y).dot(x)

    grad = grad + (_lambda / m) * temp

    return J, grad


### One vs All

In [154]:
def one_vs_all(x, y, num_labels, _lambda):
    m , n = x.shape

    all_theta = np.zeros((num_labels, n + 1))

    x = np.concatenate([np.ones((m ,1)), x], axis=1)
    
    resultado = True

    for c in np.arange(num_labels):
        initial_theta = np.zeros((n + 1))
        options = {'maxiter': 10}
        res = optimize.minimize(costo,
                                initial_theta,
                                (x, (y == c), _lambda),
                                jac=True,
                                method='CG',
                                options = options)
        all_theta[c] = res.x
        resultado = res.success

    return all_theta, resultado


In [155]:
# opteniendo los parametros optimizados
_lambda = 0.1
num_labels = 3
all_theta, resultado = one_vs_all(x_norm, y, num_labels, _lambda)

print(all_theta.shape)
print(resultado)

(3, 7)
True


In [156]:
print(all_theta)

[[-12.4381   0.      -0.       0.      -0.      -0.       0.    ]
 [ -1.614   -1.6468  -6.6423   2.8954  -1.5599  -0.4802  -2.125 ]
 [ -3.1527  -0.437    0.6563  -3.4995   2.6983   2.6111   0.5348]]


### Prediccion one vs all


In [157]:
def prediccion(theta, x):
    m = x.shape[0]
    num_labels = theta.shape[0]

    p = np.zeros(m)

    x = np.concatenate([np.ones((m, 1)), x], axis=1)
    p = np.argmax(sigmoid(x.dot(theta.T)), axis=1)

    return p



In [158]:
# probando
x1 = x_norm[275:285, : ].copy()
p = prediccion(all_theta, x1)
print(p)

[1 1 1 2 1 1 1 2 1 1]


In [159]:
# porcentaje de prediccion
pred = prediccion(all_theta, x_norm)
print('Precision del conjuto de entrenamiento: {:.2f}%'.format(np.mean(pred == y) * 100))

Precision del conjuto de entrenamiento: 79.58%
